In [1]:
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split

# Carregar dados (subindo dois níveis: DataPrep -> Code -> infnet-25E1_3, depois indo para Data/Raw)
df_dev = pd.read_parquet("../../Data/Raw/dataset_kobe_dev.parquet")
df_prod = pd.read_parquet("../../Data/Raw/dataset_kobe_prod.parquet")

# Filtrar colunas e remover NAs
cols = ['lat', 'lon', 'minutes_remaining', 'period', 'playoffs', 'shot_distance', 'shot_made_flag']
df_dev_filtered = df_dev[cols].dropna()
df_prod_filtered = df_prod[cols].dropna()

# Salvar dados processados (subindo dois níveis e indo para Data/Processed)
df_dev_filtered.to_parquet("../../Data/Processed/data_filtered.parquet")

# Dividir em treino e teste estratificado
X = df_dev_filtered.drop('shot_made_flag', axis=1)
y = df_dev_filtered['shot_made_flag']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Salvar splits
X_train.join(y_train).to_parquet("../../Data/Processed/base_train.parquet")
X_test.join(y_test).to_parquet("../../Data/Processed/base_test.parquet")

# Registrar no MLFlow
with mlflow.start_run(run_name="PreparacaoDados"):
    mlflow.log_param("test_size", 0.2)
    mlflow.log_metric("train_size", X_train.shape[0])
    mlflow.log_metric("test_size", X_test.shape[0])
